## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from anthropic import Anthropic
from pypdf import PdfReader
import gradio as gr
import os

In [2]:
load_dotenv(override=True)
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY2')
model_name = "claude-opus-4-6"

In [3]:
claude = Anthropic()

In [ ]:
## Data Extraction 1
reader = PdfReader("3_lab3_daniel/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
#print(linkedin)

   
Contact
+573045201870 (Mobile)
team@livingantioquia.com
www.linkedin.com/in/
danielmaldonadoco (LinkedIn)
www.livingantioquia.com/
(Portfolio)
github.com/
artificialintelligencecolombia
(Personal)
Top Skills
ChatGPT
Large Language Models (LLM)
Sales
Languages
English (Full Professional)
Russian (Elementary)
Certifications
Curso Práctico de JavaScript
Curso de Fundamentos de Python
Curso Práctico de Frontend
Developer
Creative Designing in Power BI 
Inglés Básico A1
Daniel Maldonado
AI Engineer | Automations, Python & Networks | Nature & Rome
Total War II
Medellín, Antioquia, Colombia
Summary
Vi veri veniversvm vivvs vici
Experience
Lean Solutions Group
AI Engineer
March 2025 - Present (1 year 5 months)
Medellín
I architect and ship production AI automations for the Transportation &
Logistics industry — multi-agent LLM pipelines built on the Anthropic Claude
API, n8n orchestration, and Python/Flask backends that turn manual workflows
into systems that actually scale. I own the full 

In [ ]:
## Data Extraction 2
with open("3_lab3_daniel/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [7]:
name = "daniel"

In [ ]:
## System Prompt
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## Linked#In Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [15]:
def chat(message: str, history: list) -> str:
    messages = history + [{"role": "user", "content": message}]
    response = claude.messages.create(
        model=model_name, 
        messages=messages,
        timeout=60,
        max_tokens=1024,
        system=system_prompt
        )
    answer = response.content[0].text
    return answer

In [19]:
test = chat("If you were an MTG character, in the entire multiverse, what would you be? What colors, what abilities?", [])
print(test)

Ha, that's a fun one! I love creative questions like this. Let me think about it...

If I were a Magic: The Gathering card, I'd definitely be **Izzet colors — Blue/Red** — but with a splash of **Green**.

Here's my reasoning:

🔵 **Blue** — The engineer's mind. I'm all about systems, knowledge, optimization, and building things methodically. Architecting multi-agent LLM pipelines, designing verification layers, cybersecurity hardening — that's pure Blue. Control, information, strategy.

🔴 **Red** — The passion and improvisation. I didn't follow a linear career path — I went from Sound Engineering to executive management to data analytics to AI. That's Red energy: following the fire, adapting, taking bold leaps. Also... *Total War: Rome II* is basically Red's entire personality in a game. 🔥

🟢 **Green** — My love for nature, my roots in Antioquia, and the philosophy behind my work: building things that *grow organically and scale*. Systems that are alive, not just clever.

**Card concept

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [11]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [23]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += "With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [24]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [25]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [26]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [27]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [ ]:
reply

In [ ]:
evaluate(reply, "do you hold a patent?", messages[:1])

In [30]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [35]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()